In [1]:
import pyvisa

VISA_ADDR = "GPIB0::17::INSTR"

rm = pyvisa.ResourceManager()
inst = rm.open_resource(VISA_ADDR)
inst.timeout = 10000

# 关键：GPIB 的 EOI 行为（默认一般就是 True，但我们显式写出来）
inst.send_end = True

print("IDN:", inst.query("*IDN?").strip())

IDN: Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401


In [2]:
def try_query(tag):
    try:
        ans = inst.query("*IDN?")
        print(tag, "raw:", repr(ans), "| strip:", repr(ans.strip()))
    except Exception as e:
        print(tag, "ERROR:", type(e).__name__, repr(e))

# A: 完全依赖 EOI（read/write termination 都不设）
inst.write_termination = None
inst.read_termination = None
try_query("A EOI-only")

# B: LF
inst.write_termination = "\n"
inst.read_termination = "\n"
try_query("B LF")

# C: CRLF
inst.write_termination = "\r\n"
inst.read_termination = "\r\n"
try_query("C CRLF")

A EOI-only raw: 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401\r\n' | strip: 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401'
B LF raw: 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401\r' | strip: 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401'
C CRLF raw: 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401' | strip: 'Agilent Technologies,B1500A,MY55231213,A.06.02.2023.0401'
